## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere -- same convention as DE_loopV1.ipynb /
# DEEPMLP_nnx_recovery.ipynb.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sequence_classesV1 import *
from analysisV1 import *
from initialize_weights import (
    load_F_viab_aav9_mlp, load_J_viab_aav9_mlp,
    initialize_correlated_weights, initialize_anticorrelated_weights, initialize_indep_weights,
    NUM_AMINO_ACIDS, NUM_POSITIONS,
)

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## Part 1 — Intro: the MLP, as trained in `AAV9_profile_model.ipynb`

`AAV9_profile_model.ipynb` (`Modelization_V1/MLP_regression/AAV_viability_test/`) trains a
small MLP (`ProfileMLP`: one-hot per-position sequence -> 2 hidden layers -> scalar score) on
`aav9.csv`, 68,776 real AAV9 capsid variants with a measured log-enrichment `target`. Two
things get pulled out of that trained network afterward:

- `F_viab` — a `(num_amino_acids=20, num_positions=7)` profile, from a SINGLE-mutant scan:
  for a batch of real background sequences, flip one position at a time and see how the
  model's predicted score shifts.
- `J_viab` — a `(7, 7, 20, 20)` pairwise coupling, from the same idea extended to a
  DOUBLE-mutant scan (flip two positions at once).

Both are exported to `.npy` (gitignored, a derived artifact) and loaded here via
`initialize_weights.py` — see that notebook for the full training/extraction code and
diagnostics (naive-vs-MLP agreement, etc.); this notebook starts from the result and treats
`F_viab`/`J_viab` as "ground truth" for what a REAL viability landscape looks like.

In [ ]:
F_viab = load_F_viab_aav9_mlp()
J_viab = load_J_viab_aav9_mlp()

print(f"F_viab shape: {F_viab.shape}")
print(f"J_viab shape: {J_viab.shape}")

_ = plot_teacher_weights(F_viab, J_viab, title="F_viab / J_viab -- real AAV9 (ProfileMLP recovery)")
plt.show()

## Shared protocol setup (used by parts 2, 3 and 4)

One `ProtocolV2` recipe, copied from `DEEPMLP_nnx_recovery.ipynb`'s toy library
(`N0=1e9`, `N1=5e8`, `dilution_factor=10`, `D=1e9`, `noise_viab=noise_sel=0.5`,
`T_viab=T_sel=0.5`) — but a SINGLE protocol here (DEEPMLP built 5 to average over training
noise; we only need one per regime). `sequences`, `F_viab` and `J_viab` are fixed once and
reused identically across parts 2-4 -- only `F_sel`/`J_sel` change between them, so any
difference in outcome is attributable to the correlation regime alone, not to a different
library or teacher model.

In [ ]:
key = jax.random.key(0)
key, k_seq = jax.random.split(key)

N = 200_000
sequences = jax.random.randint(k_seq, shape=(N, NUM_POSITIONS), minval=0, maxval=NUM_AMINO_ACIDS)


def build_protocol(F_sel, J_sel):
    """Same protocol recipe as DEEPMLP_nnx_recovery.ipynb -- a single instance here, reused
    with different F_sel/J_sel across parts 2-4."""
    return ProtocolV2(
        N0=1_000_000_000, N1=500_000_000, dilution_factor=10, sequences=sequences, D=1e9,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=0.5, noise_sel=0.5, T_sel=0.5, T_viab=0.5,
    )


def compare_F(F_viab, F_sel, title):
    """Same 3-panel recipe as AAV9_profile_model.ipynb's F comparisons: two heatmaps
    (shared RdBu_r scale) + a raw-entry scatter with Pearson r."""
    F_viab, F_sel = np.asarray(F_viab), np.asarray(F_sel)
    vmax = max(np.abs(F_viab).max(), np.abs(F_sel).max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, F, panel_title in [(axes[0], F_viab, "F_viab"), (axes[1], F_sel, "F_sel")]:
        im = ax.imshow(F, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_title(panel_title)
        ax.set_xlabel("Position"); ax.set_ylabel("Amino acid")
        ax.set_xticks(range(NUM_POSITIONS)); ax.set_xticklabels(range(1, NUM_POSITIONS + 1))
        ax.set_yticks(range(NUM_AMINO_ACIDS)); ax.set_yticklabels(AA_LABELS)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    r = pearson(F_viab.ravel(), F_sel.ravel())
    axes[2].scatter(F_viab.ravel(), F_sel.ravel(), s=14, alpha=0.6)
    lims = [min(F_viab.min(), F_sel.min()), max(F_viab.max(), F_sel.max())]
    axes[2].plot(lims, lims, "k--", alpha=0.5)
    axes[2].set_xlabel("F_viab"); axes[2].set_ylabel("F_sel")
    axes[2].set_title(f"Pearson r = {r:.3f}")

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def run_and_summarize(name, F_sel, J_sel):
    """Builds the shared protocol with this regime's F_sel/J_sel, shows F_sel vs F_viab,
    runs one loop_DE() round, and plots viability-vs-selectivity score colored by abundance
    at every pipeline stage (plot_all_lambda_scores). Returns a one-row summary dict."""
    compare_F(F_viab, F_sel, f"{name} -- F_sel vs F_viab")

    protocol = build_protocol(F_sel, J_sel)
    viab_scores = np.array(protocol.compute_score(protocol.F_viab, protocol.J_viab))
    sel_scores  = np.array(protocol.compute_score(protocol.F_sel,  protocol.J_sel))
    score_corr  = pearson(viab_scores, sel_scores)
    p10         = precision_at_k(viab_scores, sel_scores, k_frac=0.10)

    protocol.loop_DE()
    lambda4 = np.array(protocol.lambda4)

    _ = plot_all_lambda_scores(protocol, title=f"{name} -- viability vs selectivity score, by pipeline stage")
    plt.show()

    return dict(regime=name, score_corr=score_corr, top10pct_overlap=p10,
                surviving=int((lambda4 > 0).sum()), pct_surviving=100 * (lambda4 > 0).mean(),
                entropy=shannon_entropy(lambda4))

## Part 2 — Correlated weights

`F_sel`/`J_sel` share `F_viab`/`J_viab`'s trend (`correlation=0.8`): an amino acid that's good
for viability tends to also be good for selectivity. Expect a diagonal-shaped survivor region
(high viability AND high selectivity) and comparatively gentle attrition.

In [ ]:
F_sel_corr, J_sel_corr = initialize_correlated_weights(jax.random.key(10), F_viab, J_viab, correlation=0.8)
summary_corr = run_and_summarize("Correlated (r=+0.8)", F_sel_corr, J_sel_corr)
summary_corr

## Part 3 — Anticorrelated weights

`F_sel`/`J_sel` share the OPPOSITE trend (`anticorrelation=0.8`): an amino acid that's good
for viability tends to be BAD for selectivity. Expect survivors squeezed into a corner
(moderate on both axes, since being extreme on one now works against the other) and a much
harsher combined bottleneck.

In [ ]:
F_sel_anti, J_sel_anti = initialize_anticorrelated_weights(jax.random.key(11), F_viab, J_viab, anticorrelation=0.8)
summary_anti = run_and_summarize("Anticorrelated (r=-0.8)", F_sel_anti, J_sel_anti)
summary_anti

## Part 4 — Indep weights

`F_sel`/`J_sel` come from relabeling amino-acid identity in `F_viab`/`J_viab` (plus a bit of
noise) -- same overall "shape" as a real profile/coupling model, but no relationship to
`F_viab`/`J_viab`'s specific amino-acid assignments. Expect no diagonal/anti-diagonal
structure -- survivors scattered without a clear trend between the two axes.

In [ ]:
F_sel_indep, J_sel_indep = initialize_indep_weights(jax.random.key(12), F_viab, J_viab)
summary_indep = run_and_summarize("Indep (relabeled + noise)", F_sel_indep, J_sel_indep)
summary_indep

## Comparison

In [ ]:
summary = pd.DataFrame([summary_corr, summary_anti, summary_indep]).set_index("regime")
summary